In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import login
login()

In [11]:
# parameters
Tokenizer_name = "gpt2"
max_prompt_len = 32
max_response_len = 64
torch.random.manual_seed(1082)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dim = 64
num_layers = 2
num_heads = 4
stablemax = False
batch_size = 16
num_epochs = 10


In [12]:
# Model functions
def truncated_normal_(tensor, std=1.0, trunc=2.0):
    size = tensor.shape
    tmp = tensor.new_empty(size + (4,)).normal_()
    valid = (tmp < trunc) & (tmp > -trunc)
    ind = valid.max(-1, keepdim=True)[1]
    tensor.data.copy_(tmp.gather(-1, ind).squeeze(-1))
    tensor.data.mul_(std)
    return tensor

# Rotary Positional Encoding
class RotaryPositionalEncoding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(0, max_seq_len, dtype=torch.float)
        sinusoid_inp = torch.ger(positions, inv_freq)
        self.register_buffer("sin", torch.sin(sinusoid_inp))
        self.register_buffer("cos", torch.cos(sinusoid_inp))

    def forward(self, x):
        seq_len = x.size(1)
        sin = self.sin[:seq_len, :].unsqueeze(0)
        cos = self.cos[:seq_len, :].unsqueeze(0)
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        x[..., ::2] = x1 * cos - x2 * sin
        x[..., 1::2] = x1 * sin + x2 * cos
        return x

# Gated Linear Unit
class GLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim * 2, bias=False)

    def forward(self, x):
        x_proj = self.proj(x)
        a, b = x_proj.chunk(2, dim=-1)
        return a * torch.sigmoid(b)

# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.dim = dim

    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) * math.sqrt(self.dim)
        return x / (norm + self.eps)

# Transformer Block Encoder-only
class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, bias=False, batch_first=True)
        self.rmsnorm1 = RMSNorm(dim)
        self.mlp = nn.Sequential(
            GLU(dim),
            nn.Linear(dim, dim, bias=False)
        )
        self.rmsnorm2 = RMSNorm(dim)
        self.rotary_pe = RotaryPositionalEncoding(dim)

    def forward(self, x):
        x = self.rotary_pe(x)
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = x + attn_out
        x = self.rmsnorm1(x)
        mlp_out = self.mlp(x)
        x = x + mlp_out
        x = self.rmsnorm2(x)
        return x

# Embedding Layer
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim)

    def forward(self, x):
        return self.embedding(x)

# Output Head (Softmax/Stablemax)
class OutputHead(nn.Module):
    def __init__(self, dim, vocab_size, stablemax=False):
        super().__init__()
        self.linear = nn.Linear(dim, vocab_size, bias=False)
        self.stablemax = stablemax

    def forward(self, x):
        logits = self.linear(x)
        if self.stablemax:
            logits = logits - logits.max(dim=-1, keepdim=True)[0]
            exp_logits = torch.exp(logits)
            probs = exp_logits / exp_logits.sum(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits, dim=-1)
        return probs

# Hierarchical Reasoning Model
class HRMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, dim, num_layers=2, num_heads=4, stablemax=False):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, dim)
        self.high_level = nn.Sequential(
            *[EncoderBlock(dim, num_heads) for _ in range(num_layers)]
        )
        self.low_level = nn.Sequential(
            *[EncoderBlock(dim, num_heads) for _ in range(num_layers)]
        )
        self.output_head = OutputHead(dim, vocab_size, stablemax=stablemax)
        self.z0 = nn.Parameter(torch.zeros(1, dim), requires_grad=False)
        truncated_normal_(self.z0, std=1.0, trunc=2.0)

    def forward(self, src):
        x = self.embedding(src)
        x = x + self.z0
        x = self.high_level(x)
        x = self.low_level(x)
        probs = self.output_head(x)
        return probs

# Loss function
def seq2seq_loss(probs, targets):
    batch, src_seq_len, vocab_size = probs.size()
    tgt_seq_len = targets.size(1)
    seq_len = min(src_seq_len, tgt_seq_len)
    probs = probs[:, :seq_len, :]
    targets = targets[:, :seq_len]
    probs = probs.reshape(-1, vocab_size)
    targets = targets.reshape(-1)
    log_probs = torch.log(probs[torch.arange(probs.size(0)), targets] + 1e-8)
    loss = -log_probs.mean()
    return loss

In [13]:
# Load the dataset
print("Loading tiny-codes dataset...")
ds = load_dataset("nampdn-ai/tiny-codes", split="train")

Loading tiny-codes dataset...


In [14]:
# Use GPT2 tokenizer for token IDs
tokenizer = AutoTokenizer.from_pretrained(Tokenizer_name)
tokenizer.pad_token = tokenizer.eos_token
vocab_size = tokenizer.vocab_size
# Prepare batches for all languages
inputs = []
targets = []
languages = []

In [15]:
for row in ds:
    prompt = row["prompt"]
    response = row["response"]
    lang = row["programming_language"]
    # Tokenize and pad/truncate
    prompt_ids = tokenizer(prompt, padding="max_length", truncation=True, max_length=max_prompt_len, return_tensors="pt").input_ids.squeeze()
    response_ids = tokenizer(response, padding="max_length", truncation=True, max_length=max_response_len, return_tensors="pt").input_ids.squeeze()
    inputs.append(prompt_ids)
    targets.append(response_ids)
    languages.append(lang)

# Convert to tensors
src = torch.stack(inputs)  # [batch, max_prompt_len]
tgt = torch.stack(targets) # [batch, max_response_len]

In [16]:
# Model setup and training
model = HRMSeq2Seq(vocab_size, dim, num_layers, num_heads, stablemax).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
src = src.to(device)
tgt = tgt.to(device)

In [17]:
# Training loop
print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    num_batches = src.size(0) // batch_size
    for i in range(num_batches):
        batch_src = src[i*batch_size:(i+1)*batch_size]
        batch_tgt = tgt[i*batch_size:(i+1)*batch_size]
        optimizer.zero_grad()
        probs = model(batch_src)
        loss = seq2seq_loss(probs, batch_tgt)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1}/{num_epochs} - Avg Loss: {avg_loss:.4f}")

print("Training complete.")

Starting training...
Epoch 1/10 - Avg Loss: 4.3320
Epoch 2/10 - Avg Loss: 3.5994
Epoch 3/10 - Avg Loss: 3.4325
Epoch 4/10 - Avg Loss: 3.3518
Epoch 5/10 - Avg Loss: 3.3018
Epoch 6/10 - Avg Loss: 3.2669
Epoch 7/10 - Avg Loss: 3.2410
Epoch 8/10 - Avg Loss: 3.2201
Epoch 9/10 - Avg Loss: 3.2031
Epoch 10/10 - Avg Loss: 3.1881
Training complete.


In [18]:
# Save model weights
torch.save(model.state_dict(), "hrmseq2seq.pt")
print("Model saved to hrmseq2seq.pt")

Model saved to hrmseq2seq.pt
